# Spyglass environment smoke test

Run this notebook top to bottom (**Run → Run All Cells**) to check that this server has what the Spyglass tutorial needs:

1. the Python packages from `binder/environment.yml` are installed,
2. the MySQL database inside this container is running and DataJoint can log in,
3. the main Spyglass modules import (most of them create their database tables when imported),
4. the tutorial NWB file is on the shared data drive and can be opened.

Each section ends with a ✅ or ❌ line. The last cell gives a summary.

In [ ]:
import os, sys, time, platform
from pathlib import Path

RESULTS = {}  # check name -> (ok, detail)

def record(name, ok, detail=""):
    RESULTS[name] = (ok, detail)
    print(("✅" if ok else "❌"), name, ("— " + detail) if detail else "")

print("Python  ", sys.version.split()[0], "at", sys.executable)
print("Platform", platform.platform())

def cgroup_value(*paths):
    for p in paths:
        try:
            return Path(p).read_text().strip()
        except OSError:
            pass
    return None

mem = cgroup_value("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory/memory.limit_in_bytes")
if mem and mem.isdigit() and int(mem) < 1 << 60:
    print(f"Memory limit {int(mem) / 2**30:.2f} GiB")
else:
    print("Memory limit: none detected")

## 1. Installed packages

These are the packages named in `binder/environment.yml`, plus a few that Spyglass pulls in. The *import* column is the module name used in Python.

In [ ]:
from importlib import metadata

# (distribution name, import name)
REQUIREMENTS = [
    ("spyglass-neuro", "spyglass"),
    ("datajoint", "datajoint"),
    ("pynwb", "pynwb"),
    ("hdmf", "hdmf"),
    ("h5py", "h5py"),
    ("ndx-franklab-novela", "ndx_franklab_novela"),
    ("spikeinterface", "spikeinterface"),
    ("mountainsort4", "mountainsort4"),
    ("sortingview", "sortingview"),
    ("ghostipy", "ghostipy"),
    ("pyfftw", "pyfftw"),
    ("non_local_detector", "non_local_detector"),
    ("position_tools", "position_tools"),
    ("ripple_detection", "ripple_detection"),
    ("track_linearization", "track_linearization"),
    ("numpy", "numpy"),
    ("scipy", "scipy"),
    ("pandas", "pandas"),
    ("xarray", "xarray"),
    ("matplotlib", "matplotlib"),
    ("seaborn", "seaborn"),
    ("bottleneck", "bottleneck"),
    ("panel", "panel"),
    ("pubnub", "pubnub"),
    ("pydotplus", "pydotplus"),
    ("ipympl", "ipympl"),
    ("opencv-python", "cv2"),
    ("black", "black"),
    ("pymysql", "pymysql"),
]

def dist_version(name):
    for candidate in (name, name.replace("_", "-"), name.replace("-", "_")):
        try:
            return metadata.version(candidate)
        except metadata.PackageNotFoundError:
            pass
    return None

missing = []
print(f"{'package':<22} {'version':<14} import")
for dist, mod in REQUIREMENTS:
    version = dist_version(dist)
    try:
        __import__(mod)
        imported = "ok"
    except Exception as e:  # report, don't stop
        imported = f"FAILED: {type(e).__name__}: {e}"[:80]
    if version is None or imported != "ok":
        missing.append(dist)
    print(f"{dist:<22} {version or '-':<14} {imported}")

record("packages", not missing, "all present" if not missing else "problems: " + ", ".join(missing))

## 2. Database

Spyglass keeps its tables in MySQL. On this hub a MySQL 8.0 server runs inside your own container: the `start` script launches it and it only listens on `127.0.0.1`. The DataJoint connection settings are in `~/.datajoint_config.json`.

The database lives only in this container. **Everything in it is lost when the server stops** (after 1 hour idle).

In [ ]:
import datajoint as dj

print("config file:", Path("~/.datajoint_config.json").expanduser())
print("host:", dj.config["database.host"], "user:", dj.config["database.user"])
try:
    conn = dj.conn(reset=True)
    version = conn.query("SELECT VERSION()").fetchone()[0]
    schemas = [s for (s,) in conn.query("SHOW DATABASES")
               if s not in ("information_schema", "mysql", "performance_schema", "sys")]
    record("database", True, f"MySQL {version}; {len(schemas)} Spyglass schemas so far")
except Exception as e:
    record("database", False, f"{type(e).__name__}: {e}. See ~/.spyglass-mysql/mysqld.log")

## 3. Import the main Spyglass modules

Importing a pipeline module declares its tables in the database, so the first run on a fresh server takes longer than later runs.

In [ ]:
import spyglass
from spyglass.settings import config as sg_config

print("spyglass", spyglass.__version__, "from", Path(spyglass.__file__).parent)
print("raw dir:", sg_config.get("SPYGLASS_RAW_DIR"))
print("analysis dir:", sg_config.get("SPYGLASS_ANALYSIS_DIR"))

MODULES = [
    "spyglass.common",
    "spyglass.lfp",
    "spyglass.position",
    "spyglass.linearization",
    "spyglass.spikesorting.v1",
    "spyglass.ripple.v1",
    "spyglass.decoding.v1",
]
failed = []
for name in MODULES:
    t0 = time.time()
    try:
        __import__(name)
        print(f"  ok      {name:<28} {time.time() - t0:5.1f} s")
    except Exception as e:
        failed.append(name)
        print(f"  FAILED  {name:<28} {type(e).__name__}: {e}"[:160])

record("spyglass imports", not failed,
       f"{len(MODULES)} modules" if not failed else "failed: " + ", ".join(failed))

In [ ]:
# A real query against a table that importing spyglass.common just created.
import spyglass.common as sgc
print(sgc.Nwbfile())
record("table query", True, f"Nwbfile has {len(sgc.Nwbfile())} rows")

## 4. Tutorial data on the shared drive

The admins copy the tutorial NWB file to the read-only shared folder (`$SHARED_DATA_DIR`, normally `/home/jovyan/data`). At start-up, every `*.nwb` file in that folder (or in its `spyglass/raw/` subfolder) is linked into Spyglass's raw-data folder, which is writable.

In [ ]:
NWB_NAME = "minirec20230622.nwb"   # the tutorial's "minimal recording", ~240 MB

shared = Path(os.environ.get("SHARED_DATA_DIR", "/home/jovyan/data"))
raw_dir = Path(sg_config.get("SPYGLASS_RAW_DIR", "~/spyglass_data/raw")).expanduser()
print("SHARED_DATA_DIR =", shared, "(exists)" if shared.is_dir() else "(MISSING)")

found = [p for p in (shared / "spyglass" / "raw" / NWB_NAME, shared / NWB_NAME) if p.is_file()]
if found:
    nwb_path = found[0]
    size_mb = nwb_path.stat().st_size / 1e6
    print(f"found  {nwb_path}  ({size_mb:.0f} MB)")
    linked = raw_dir / NWB_NAME
    print(f"linked {linked} -> {os.path.realpath(linked)}" if linked.exists()
          else f"not linked into {raw_dir} (start script did not run?)")
    record("NWB on shared drive", True, f"{nwb_path} ({size_mb:.0f} MB)")
else:
    nwb_path = None
    others = sorted(str(p) for p in shared.rglob("*.nwb"))[:20] if shared.is_dir() else []
    print("NWB files present:", others or "none")
    record("NWB on shared drive", False,
           f"{NWB_NAME} not found in {shared} or {shared / 'spyglass/raw'}")

In [ ]:
# Open the file read-only and print a little metadata (reads only the header, not the data).
if nwb_path is not None:
    import pynwb
    try:
        with pynwb.NWBHDF5IO(str(nwb_path), "r", load_namespaces=True) as io:
            nwb = io.read()
            print("identifier: ", nwb.identifier)
            print("session:    ", nwb.session_description)
            print("start time: ", nwb.session_start_time)
            print("subject:    ", getattr(nwb.subject, "subject_id", None))
            print("electrodes: ", len(nwb.electrodes) if nwb.electrodes is not None else 0)
            print("acquisition:", list(nwb.acquisition))
        record("NWB readable", True, nwb.identifier)
    except Exception as e:
        record("NWB readable", False, f"{type(e).__name__}: {e}")
else:
    record("NWB readable", False, "skipped: file not found")

## Summary

In [ ]:
print(f"{'check':<22} result")
for name, (ok, detail) in RESULTS.items():
    print(f"{name:<22} {'✅ pass' if ok else '❌ FAIL'}  {detail}")
if all(ok for ok, _ in RESULTS.values()):
    print("\nAll checks passed: this server is ready for the Spyglass tutorial.")
else:
    print("\nSome checks failed; see the sections above.")